# ClipForge — قالب فيديو ثابت للهاتف

هذا القالب يعالج الفيديو تلقائيًا خارج المتصفح باستخدام FFmpeg. النتيجة: فيديو عمودي 9:16 بدقة 1080×1920، تحسين ألوان خفيف، نص اختياري، ولوجو اختياري. لا تحتاج لتثبيت أي برنامج على الهاتف.


In [ ]:
!apt-get -qq update && apt-get -qq install -y ffmpeg


In [ ]:
from google.colab import files
from pathlib import Path
print('ارفع فيديو واحدًا من الهاتف')
up=files.upload()
video=next(iter(up))
print('ارفع Logo اختياريًا بصيغة PNG/JPG، ثم اضغط إلغاء إذا لا تريد لوجو')
try:
    up_logo=files.upload()
    logo=next(iter(up_logo)) if up_logo else ''
except Exception:
    logo=''
print('تم تجهيز القالب')


## إعدادات بسيطة اختيارية
اترك القيم كما هي لتشغيل القالب الجاهز. غيّر النص أو مكان اللوجو إذا أردت.


In [ ]:
title='ClipForge'              # اكتب '' بدون عنوان
logo_position='top-right'     # top-right أو top-left أو bottom-right أو bottom-left
logo_width=220
trim_start=0                 # ثواني
trim_end=0                   # 0 = حتى نهاية الفيديو
output='clipforge_template.mp4'


In [ ]:
import subprocess
# قالب عمودي ثابت: يملأ 9:16 من منتصف الفيديو مع ألوان محسنة
filters=['scale=1080:1920:force_original_aspect_ratio=increase','crop=1080:1920','eq=brightness=0.02:contrast=1.05:saturation=1.08']
if title:
    safe=title.replace('\\','').replace(':','\\:').replace("'","\\'")
    filters.append(f"drawtext=text='{safe}':fontcolor=white:fontsize=64:borderw=4:bordercolor=black:x=(w-text_w)/2:y=h-text_h-100")
vf=','.join(filters)
cmd=['ffmpeg','-y']
if trim_start>0: cmd += ['-ss',str(trim_start)]
cmd += ['-i',video]
if trim_end>trim_start: cmd += ['-to',str(trim_end-trim_start)]
if logo:
    positions={'top-right':'W-w-36:36','top-left':'36:36','bottom-right':'W-w-36:H-h-36','bottom-left':'36:H-h-36'}
    cmd += ['-i',logo,'-filter_complex',f'[0:v]{vf}[base];[1:v]scale={logo_width}:-1[lg];[base][lg]overlay={positions.get(logo_position,positions["top-right"])}[videoout]','-map','[videoout]','-map','0:a?']
else:
    cmd += ['-vf',vf,'-map','0:v','-map','0:a?']
cmd += ['-c:v','libx264','-preset','veryfast','-crf','22','-c:a','aac','-b:a','128k','-movflags','+faststart',output]
print('جاري تطبيق القالب الثابت...')
subprocess.run(cmd,check=True)
print('تم إنشاء:',output)


In [ ]:
from google.colab import files
files.download(output)
